Generate a chart from a preprocessed `.pt` song

Runs top to bottom: setup, load model, define the generator, then set song settings and run.

In [1]:
from pathlib import Path

from tracers.project import setup_project
from tracers.pipelines.trainer_factory import build_trainer

PROJECT_ROOT = Path.cwd().parent
setup = setup_project(PROJECT_ROOT)

MODEL_KEY = "tracer_global_np"

experiment_config = setup["experiment_config"]
hyperparams = setup["hyperparams"]
segment_frames = setup["segment_frames"]
device = setup["device"]

In [2]:
from tracers.pipelines.generate import run_custom_generation

trainer, model = build_trainer(
    model_key=MODEL_KEY,
    hyperparams=hyperparams,
    experiment_config=experiment_config,
    device=device,
    checkpoint_path=str(PROJECT_ROOT / "notebooks" / "checkpoints" / "best.pt"),
    train_loader=None,
    valid_loader=None,
)

Loaded c:\Users\joshua\vsjycode\csc173\transformers\TRACERS\notebooks\checkpoints\best.pt (epoch 39, best val loss 0.1490)


In [3]:
from tracers.export.chart import export_generated_chart

# ---------- settings ----------
AUDIO_PT_NAME = "nichijou"
AUDIO_PT = PROJECT_ROOT / f"{AUDIO_PT_NAME}.pt"
DIFFICULTY = 3
BPM = 157
OFFSET_MS = 0
LANE_MAP = (0, 2, 4, 6)

TEMPERATURE = 0.4
TOP_K = None
TOP_P = None
SEED = 0
SAVE_DEBUG_CSV = True

# ---------- generate ----------
generated, debug_segments = run_custom_generation(
    trainer,
    AUDIO_PT,
    difficulty=DIFFICULTY,
    segment_frames=segment_frames,
    context_frames=500,
    seed_action=0,
    seed=SEED,
    temperature=TEMPERATURE,
    top_k=TOP_K,
    top_p=TOP_P,
    collect_debug=SAVE_DEBUG_CSV,
)

# ---------- export ----------
output_dir = Path(f"charts/{AUDIO_PT_NAME}")
output_dir.mkdir(parents=True, exist_ok=True)

export_generated_chart(
    generated,
    output_dir / "generated_chart_global.json",
    bpm=BPM,
    start_frame=0,
    offset_ms=OFFSET_MS,
    lane_map=LANE_MAP,
)
print(f"Saved generated_chart_global.json")

if SAVE_DEBUG_CSV:
    from tracers.generate.converter import export_generation_debug
    export_generation_debug(debug_segments, output_path=str(output_dir / "generation_debug.csv"), top_k=5)

Prediction frames: 24808 (248.1 s)
Window size: 2000 | Context: 500 | Stride: 1500
Window      1-  2000 | Context:    0 | New: 2000 | Active:   12
Window   1501-  3500 | Context:  500 | New: 1500 | Active:    0
Window   3001-  5000 | Context:  500 | New: 1500 | Active:    0
Window   4501-  6500 | Context:  500 | New: 1500 | Active:    0


KeyboardInterrupt: 

In [ ]:

export_generation_debug(debug_segments, output_path=f"charts/{AUDIO_PT_NAME}/generation_debug.csv", top_k=5)

Exported 25475 frame records to: charts\renai\generation_debug.csv


25475